# Segmentación jerárquica: perfiles, subperfiles y microsegmentos

Segmentación en dos etapas para microtargeting:

1. **Microsegmentos:** KMeans ponderado con muchos grupos (50-200) sobre todas las celdas.
2. **Perfiles y subperfiles:** clustering jerárquico aglomerativo (Ward ponderado) sobre los
   centros de los microsegmentos.

El árbol se corta en 4-6 perfiles para el documento técnico y en ~15 subperfiles para campañas
focalizadas. Los niveles están anidados: cada microsegmento pertenece a un solo subperfil y cada
subperfil a un solo perfil.

Tiempo estimado: ~10 min (KMeans de hasta 200 grupos sobre 645k celdas, varias veces).

## 1. Configuración

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 2. Parametros y rutas

`VARIABLES` y `USAR_PCA` definen el espacio de clustering. Quedan en el conjunto del notebook 04
hasta decidir entre niveles (`C_trayectoria` con PCA, ganador del 05) y forma (`S_forma`, notebook 06).

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"

ELECCIONES = ["P2009", "P2012", "P2015", "P2018", "P2021", "P2024"]
CORTES_EDAD = [17, 24, 29, 39, 49, 59, 69, 79, 200]
ETIQUETAS_EDAD = ["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
FORMA = [f"F{e[1:]}" for e in ELECCIONES]

OPCIONES_VARIABLES = {
    "A_base": ["promedio", "brecha_pres", "tendencia"],
    "C_trayectoria": ELECCIONES,
    "S_forma": FORMA,
}
VARIABLES = "A_base"
USAR_PCA = False

N_MICRO_CANDIDATOS = [50, 100, 200]
R2_MICRO_MIN = 0.95
K_MACRO_RANGO = range(4, 7)  # convocatoria pide 4-6 perfiles
K_MACRO = None  # None: se elige por silhouette dentro del rango
K_SUB = 15
N_EVAL = 10_000
N_BOOT = 3
RANDOM_STATE = 42

COLORES_PERFIL = ["#3288bd", "#66c2a5", "#fee08b", "#fdae61", "#f46d43", "#d53e4f"]
EDONOM = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur",
    4: "Campeche", 5: "Coahuila", 6: "Colima", 7: "Chiapas", 8: "Chihuahua",
    9: "Ciudad de México", 10: "Durango", 11: "Guanajuato", 12: "Guerrero",
    13: "Hidalgo", 14: "Jalisco", 15: "México", 16: "Michoacán", 17: "Morelos",
    18: "Nayarit", 19: "Nuevo Leon", 20: "Oaxaca", 21: "Puebla", 22: "Queretaro",
    23: "Quintana Roo", 24: "San Luis Potosi", 25: "Sinaloa", 26: "Sonora",
    27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz",
    31: "Yucatan", 32: "Zacatecas",
}

## 3. Carga de datos

### 3.1 Celdas, peso, PJ 2025 y municipio

Mismo procedimiento que los notebooks 05 y 06 (casos completos, peso = SV + NV de 2024).
Se agrega el municipio de cada sección para la tabla territorial.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet")
celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)

fed24 = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["EDOCVE", "SECCIÓN", "SEXO", "EDAD", "AELEC", "SV", "NV", "MPIOCVE", "MPIONOM"],
    filters=[("AELEC", "==", 2024)],
)
fed24["GRUPO_EDAD"] = pd.cut(fed24["EDAD"], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD)
fed24["VOTANTES"] = fed24["SV"] + fed24["NV"]
pesos = (
    fed24.groupby(["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], observed=True)["VOTANTES"]
    .sum()
    .reset_index()
    .rename(columns={"VOTANTES": "peso"})
)
pesos["GRUPO_EDAD"] = pesos["GRUPO_EDAD"].astype(str)
sec_mpio = (
    fed24.groupby(["EDOCVE", "SECCIÓN"])
    .agg(MPIOCVE=("MPIOCVE", "first"), MPIONOM=("MPIONOM", "first"))
    .reset_index()
)
del fed24

secciones = pd.read_parquet(
    DIR_PREPROCESSED / "secciones_historial.parquet", columns=["EDOCVE", "SECCIÓN", "P_PJ2025"]
)
celdas = (
    celdas.merge(pesos, on=["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"], how="left")
    .merge(secciones, on=["EDOCVE", "SECCIÓN"], how="left")
    .merge(sec_mpio, on=["EDOCVE", "SECCIÓN"], how="left")
    .dropna(subset=["peso", "P_PJ2025"])
    .reset_index(drop=True)
)
del pesos, secciones, sec_mpio

celdas["MUJER"] = (celdas["SEXO"] == 1).astype(float)
celdas["RURAL_MIXTA"] = (celdas["TIPOSEC"].astype(str) != "U").astype(float)
celdas[FORMA] = celdas[ELECCIONES].to_numpy() - celdas[ELECCIONES].mean(axis=1).to_numpy()[:, None]
print(f"Celdas: {len(celdas):,}  peso: {celdas['peso'].sum():,.0f}")

### 3.2 Espacio de clustering

Estandarizacion ponderada por peso (y PCA opcional). La muestra eval (10k celdas, probabilidad
proporcional al peso) se usa para silhouette y ARI.

In [ ]:
X = celdas[OPCIONES_VARIABLES[VARIABLES]].to_numpy(dtype=float)
w = celdas["peso"].to_numpy(dtype=float)

Z = StandardScaler().fit(X, sample_weight=w).transform(X)
if USAR_PCA:
    Z = PCA(n_components=0.9, random_state=RANDOM_STATE).fit_transform(Z)

rng = np.random.default_rng(RANDOM_STATE)
idx_eval = rng.choice(len(celdas), size=N_EVAL, replace=False, p=w / w.sum())
print(f"Variables: {VARIABLES} ({Z.shape[1]} dimensiones), PCA={USAR_PCA}")

### 3.3 Funciones auxiliares

- `r2_ponderado`: proporción de la varianza (ponderada por personas) que explican los grupos.
  Es el complemento de la inercia de KMeans.
- `media_ponderada` y `dominante`: resumenes por grupo ponderados por peso.

In [ ]:
def r2_ponderado(Z, labels, w):
    wl = np.bincount(labels, weights=w)
    centros = np.stack([np.bincount(labels, weights=w * Z[:, j]) for j in range(Z.shape[1])], axis=1)
    centros = centros / wl[:, None]
    total = (w[:, None] * (Z - np.average(Z, axis=0, weights=w)) ** 2).sum()
    dentro = (w[:, None] * (Z - centros[labels]) ** 2).sum()
    return 1 - dentro / total


def media_ponderada(df, col, por):
    wx = (df[col] * df["peso"]).groupby([df[c] for c in por]).sum()
    return wx / df.groupby(por)["peso"].sum()


def dominante(df, por, col):
    t = df.groupby(por + [col])["peso"].sum()
    return t.groupby(level=por).idxmax().map(lambda x: x[-1])

## 4. Microsegmentos

### 4.1 Número de microsegmentos

KMeans ponderado con 50, 100 y 200 grupos. Se elige el menor número que conserve al menos 95%
de la varianza (R2 >= 0.95): con eso, trabajar con los centros en lugar de las celdas pierde poca
información.

In [ ]:
micro_modelos, filas = {}, []
for n in N_MICRO_CANDIDATOS:
    km = KMeans(n, n_init=3, random_state=RANDOM_STATE).fit(Z, sample_weight=w)
    micro_modelos[n] = km
    tam = np.bincount(km.labels_, weights=w, minlength=n) / w.sum() * 100
    filas.append({"n_micro": n, "r2": r2_ponderado(Z, km.labels_, w),
                  "pct_min": tam.min(), "pct_max": tam.max()})

df_micro = pd.DataFrame(filas)
ok = df_micro[df_micro["r2"] >= R2_MICRO_MIN]
N_MICRO = int(ok["n_micro"].min()) if len(ok) else max(N_MICRO_CANDIDATOS)
print(f"N_MICRO elegido: {N_MICRO}")
df_micro.round(4)

### 4.2 Centros y peso de cada microsegmento

In [ ]:
micro = micro_modelos[N_MICRO].labels_
centros_micro = micro_modelos[N_MICRO].cluster_centers_
peso_micro = np.bincount(micro, weights=w, minlength=N_MICRO)
print(f"{N_MICRO} microsegmentos; peso mediano {np.median(peso_micro):,.0f} personas")

## 5. Árbol de perfiles (Ward ponderado)

### 5.1 Implementación

El Ward de scipy trata cada punto con el mismo peso, pero cada microsegmento representa un número
distinto de personas. Aquí, en cada paso se unen los dos grupos con menor costo
`w_a * w_b / (w_a + w_b) * ||c_a - c_b||^2`, que es el aumento de inercia de KMeans al fusionarlos.
La salida tiene el formato de `linkage` de scipy para usar `fcluster` y `dendrogram`.

In [ ]:
def ward_ponderado(C, pesos):
    n = len(C)
    centros = {i: C[i] for i in range(n)}
    masa = {i: p for i, p in enumerate(pesos / pesos.sum())}
    conteo = {i: 1 for i in range(n)}
    activos, nuevo, filas = list(range(n)), n, []
    while len(activos) > 1:
        A = np.array([centros[i] for i in activos])
        W = np.array([masa[i] for i in activos])
        d2 = ((A[:, None, :] - A[None, :, :]) ** 2).sum(axis=-1)
        costo = W[:, None] * W[None, :] / (W[:, None] + W[None, :]) * d2
        np.fill_diagonal(costo, np.inf)
        a, b = np.unravel_index(np.argmin(costo), costo.shape)
        i, j = activos[a], activos[b]
        filas.append([min(i, j), max(i, j), np.sqrt(2 * costo[a, b]), conteo[i] + conteo[j]])
        m = masa[i] + masa[j]
        centros[nuevo] = (masa[i] * centros[i] + masa[j] * centros[j]) / m
        masa[nuevo], conteo[nuevo] = m, conteo[i] + conteo[j]
        activos = [x for x in activos if x not in (i, j)] + [nuevo]
        nuevo += 1
    return np.array(filas)


def cortar(L, k):
    return fcluster(L, k, criterion="maxclust") - 1

### 5.2 Validación

Con pesos iguales, la implementación debe dar la misma particion que el Ward de scipy (ARI = 1).

In [ ]:
prueba = centros_micro[:40]
ari_prueba = adjusted_rand_score(
    cortar(ward_ponderado(prueba, np.ones(len(prueba))), 5),
    cortar(linkage(prueba, method="ward"), 5),
)
print(f"ARI contra scipy (40 puntos, k=5): {ari_prueba:.3f}")

### 5.3 Número de perfiles

Para k = 4-6: R2 ponderado sobre las celdas, silhouette sobre la muestra eval y tamaño del perfil
más chico. El salto de altura entre fusiones consecutivas indica cortes naturales del árbol
(mayor salto, corte más claro).

In [ ]:
L = ward_ponderado(centros_micro, peso_micro)
alturas = L[::-1, 2]

filas = []
for k in K_MACRO_RANGO:
    macro_k = cortar(L, k)[micro]
    tam = np.bincount(macro_k, weights=w) / w.sum() * 100
    filas.append({
        "k": k,
        "r2": r2_ponderado(Z, macro_k, w),
        "silhouette": silhouette_score(Z[idx_eval], macro_k[idx_eval]),
        "pct_min": tam.min(),
        "salto_altura": alturas[k - 2] - alturas[k - 1],
    })
df_k = pd.DataFrame(filas)
if K_MACRO is None:
    K_MACRO = int(df_k.loc[df_k["silhouette"].idxmax(), "k"])
print(f"K_MACRO: {K_MACRO}")
df_k.round(4)

### 5.4 Dendrograma

Las hojas son los subperfiles (entre parentesis, cuantos microsegmentos contienen). La linea
punteada marca el corte en perfiles.

In [ ]:
umbral = (alturas[K_MACRO - 2] + alturas[K_MACRO - 1]) / 2

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(L, truncate_mode="lastp", p=K_SUB, color_threshold=umbral, show_leaf_counts=True, ax=ax)
ax.axhline(umbral, color="gray", linestyle="--", linewidth=1)
ax.set_title(f"Árbol de microsegmentos: {K_SUB} subperfiles, corte en {K_MACRO} perfiles")
ax.set_ylabel("Altura de fusión (Ward ponderado)")
plt.tight_layout()
plt.show()

## 6. Tres niveles

### 6.1 Asignación

Cada celda hereda subperfil y perfil de su microsegmento. Los perfiles se numeran por promedio de
participación (0 = menor), igual que en el notebook 04. Los subperfiles se numeran dentro de su
perfil, también por promedio (ej. 2.1, 2.2).

In [ ]:
celdas["micro"] = micro
celdas["perfil_raw"] = cortar(L, K_MACRO)[micro]
celdas["sub_id"] = cortar(L, K_SUB)[micro]

orden = media_ponderada(celdas, "promedio", ["perfil_raw"]).sort_values().index
celdas["perfil"] = celdas["perfil_raw"].map({old: new for new, old in enumerate(orden)})

prom_sub = (
    media_ponderada(celdas, "promedio", ["perfil", "sub_id"])
    .rename("prom")
    .reset_index()
    .sort_values("prom")
)
prom_sub["subperfil"] = (
    prom_sub["perfil"].astype(str) + "." + (prom_sub.groupby("perfil").cumcount() + 1).astype(str)
)
celdas["subperfil"] = celdas["sub_id"].map(prom_sub.set_index("sub_id")["subperfil"])
celdas = celdas.drop(columns="perfil_raw")

celdas.groupby("perfil")["subperfil"].nunique().rename("subperfiles")

### 6.2 Resumen ponderado

Misma función para los tres niveles: personas, conducta (promedio, brecha, tendencia, PJ 2025),
composición (% mujer, % rural o mixta, edad y región dominantes).

In [ ]:
def resumen(df, por):
    personas = df.groupby(por)["peso"].sum()
    out = pd.DataFrame({"personas": personas, "pct_pob": personas / df["peso"].sum() * 100})
    for col in ["promedio", "brecha_pres", "tendencia", "P_PJ2025", "MUJER", "RURAL_MIXTA"]:
        out[col] = media_ponderada(df, col, por)
    out[["MUJER", "RURAL_MIXTA"]] *= 100
    out["edad_dominante"] = dominante(df, por, "GRUPO_EDAD")
    out["region_dominante"] = dominante(df, por, "REGIÓN")
    return out.round(1)


resumen(celdas, ["perfil"])

### 6.3 Subperfiles

In [ ]:
resumen(celdas, ["perfil", "subperfil"])

### 6.4 Trayectorias de los subperfiles

Un panel por perfil. Muestra si los subperfiles difieren en nivel, en forma o en ambos.

In [ ]:
anios = [2009, 2012, 2015, 2018, 2021, 2024]
tray_sub = pd.DataFrame({e: media_ponderada(celdas, e, ["perfil", "subperfil"]) for e in ELECCIONES})
pct_sub = celdas.groupby(["perfil", "subperfil"])["peso"].sum() / celdas["peso"].sum() * 100

fig, axes = plt.subplots(1, K_MACRO, figsize=(4 * K_MACRO, 4), sharey=True)
for p, ax in enumerate(axes):
    for (pp, s), fila in tray_sub.loc[[p]].iterrows():
        ax.plot(anios, fila.to_numpy(), "o-", label=f"{s} ({pct_sub[(pp, s)]:.1f}%)")
    ax.set_title(f"Perfil {p}", color=COLORES_PERFIL[p], fontweight="bold")
    ax.set_xticks(anios, [str(a)[2:] for a in anios])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel("% participación (ponderada)")
plt.tight_layout()
plt.show()

### 6.5 Tabla de microsegmentos

Base para microtargeting: cada microsegmento con su perfil, subperfil, tamaño, conducta,
composición y las 3 entidades donde tiene más personas. Se muestran los 5 más grandes por perfil.

In [ ]:
tabla_micro = resumen(celdas, ["perfil", "subperfil", "micro"])

top_ent = (
    celdas.groupby(["micro", "EDOCVE"])["peso"].sum()
    .groupby(level="micro", group_keys=False).nlargest(3)
    .reset_index()
)
top_ent["entidad"] = top_ent["EDOCVE"].map(EDONOM)
tabla_micro = tabla_micro.join(top_ent.groupby("micro")["entidad"].agg(", ".join).rename("top_entidades"))

(tabla_micro.sort_values("personas", ascending=False)
 .groupby(level="perfil").head(5)
 .sort_index())

### 6.6 Mapa de microsegmentos

Cada burbuja es un microsegmento, con tamaño proporcional a las personas que representa.

In [ ]:
t = tabla_micro.reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
for p, d in t.groupby("perfil"):
    ax.scatter(d["promedio"], d["brecha_pres"], s=d["pct_pob"] * 80, color=COLORES_PERFIL[p],
               alpha=0.75, edgecolor="#555", linewidth=0.5, label=f"Perfil {p}")
ax.set_xlabel("Promedio de participación 2009-2024 (%)")
ax.set_ylabel("Brecha presidencial (puntos)")
ax.set_title("Microsegmentos")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Estabilidad y comparación

### 7.1 Bootstrap

Bootstrap de Poisson: el peso de cada celda se multiplica por un Poisson(1), equivalente a
remuestrear personas con reemplazo. Se repiten microsegmentos y árbol, y se compara contra la
solucion original con ARI en la muestra eval.

In [ ]:
filas = []
for b in range(N_BOOT):
    w_b = w * np.random.default_rng(1000 + b).poisson(1.0, size=len(w))
    km_b = KMeans(N_MICRO, n_init=3, random_state=1000 + b).fit(Z, sample_weight=w_b)
    p_b = np.bincount(km_b.labels_, weights=w_b, minlength=N_MICRO) + 1e-9
    L_b = ward_ponderado(km_b.cluster_centers_, p_b)
    micro_eval = km_b.labels_[idx_eval]
    filas.append({
        "réplica": b + 1,
        "ari_perfil": adjusted_rand_score(celdas["perfil"].to_numpy()[idx_eval],
                                          cortar(L_b, K_MACRO)[micro_eval]),
        "ari_subperfil": adjusted_rand_score(celdas["sub_id"].to_numpy()[idx_eval],
                                             cortar(L_b, K_SUB)[micro_eval]),
    })
estabilidad = pd.DataFrame(filas)
print(f"ARI medio perfil: {estabilidad['ari_perfil'].mean():.3f}  "
      f"subperfil: {estabilidad['ari_subperfil'].mean():.3f}")
estabilidad.round(3)

### 7.2 Comparación con modelos previos

ARI de los perfiles contra el modelo del notebook 04 y los finalistas del 05, si existe
`celdas_perfiles_candidatos.parquet`.

In [ ]:
ruta = DIR_PREPROCESSED / "celdas_perfiles_candidatos.parquet"
if ruta.exists():
    llaves = ["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"]
    previos = celdas[llaves + ["perfil"]].merge(
        pd.read_parquet(ruta).drop(columns="peso"), on=llaves, how="left"
    )
    for col in [c for c in previos.columns if c.startswith(("base_", "candidato_"))]:
        otro = previos[col].fillna(-1).astype(int).to_numpy()[idx_eval]
        ari = adjusted_rand_score(previos["perfil"].to_numpy()[idx_eval], otro)
        print(f"ARI vs {col}: {ari:.3f}")
else:
    print("Sin resultados del notebook 05 para comparar.")

## 8. Territorio

Para cada municipio: microsegmento con más personas (y su subperfil y perfil), su porcentaje
dentro del municipio y la distribución de perfiles. Sirve para asignar campañas por territorio.

In [ ]:
llave_mpio = ["EDOCVE", "MPIOCVE"]
mpio = celdas.groupby(llave_mpio + ["MPIONOM", "micro"])["peso"].sum().reset_index()
mpio["pct_micro"] = mpio["peso"] / mpio.groupby(llave_mpio)["peso"].transform("sum") * 100
municipios = mpio.loc[mpio.groupby(llave_mpio)["peso"].idxmax()].copy()

niveles = celdas.drop_duplicates("micro").set_index("micro")[["perfil", "subperfil"]]
municipios = municipios.join(niveles, on="micro")

dist = celdas.groupby(llave_mpio + ["perfil"])["peso"].sum().unstack(fill_value=0)
dist = (dist.div(dist.sum(axis=1), axis=0) * 100).round(1).add_prefix("pct_perfil_")
municipios = municipios.join(dist, on=llave_mpio)
municipios["entidad"] = municipios["EDOCVE"].map(EDONOM)
municipios = municipios.rename(columns={"peso": "personas_micro"})

print(f"Municipios: {len(municipios):,}")
municipios.sort_values("pct_micro", ascending=False).head(10)

## 9. Exportacion

No se sobrescribe nada de los notebooks 04-06.

In [ ]:
salidas = {
    "celdas_jerarquia.parquet": celdas[["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD", "peso",
                                         "micro", "subperfil", "perfil"]],
    "jerarquia_perfiles.csv": resumen(celdas, ["perfil"]),
    "jerarquia_subperfiles.csv": resumen(celdas, ["perfil", "subperfil"]),
    "jerarquia_microsegmentos.csv": tabla_micro,
    "jerarquia_municipios.csv": municipios,
}
for nombre, df in salidas.items():
    ruta = DIR_PREPROCESSED / nombre
    if nombre.endswith(".parquet"):
        df.to_parquet(ruta, index=False)
    else:
        df.to_csv(ruta, index=nombre != "jerarquia_municipios.csv")
    print(f"{nombre}: {ruta.stat().st_size / 1024:.0f} KB")